In [1]:
from skyfield.api import load, wgs84
from scipy.optimize import minimize
from tqdm import tqdm
from datetime import datetime, timedelta
import pytz
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import time

In [16]:
def get_simulated_phase_difference(gt_loc, rx_param, tle_file, satellite_name, sim_start_time, sat_start_sec, duration, step = 1):
    """
    This function simulates the phase difference between the satellite and the ground station
    :param gt_loc: ground station location in lat, lon
    :param rx_param: receiver parameters (lat, lon, alpha, beta, gamma, d_10, d_12)
    :param tle_file: tle file
    :param satellite_name: satellite name
    :param sim_start_time: simulation start time
    :param sat_start_sec: satellite start time
    :param duration: duration of the simulation
    :param step: step size
    :return: time, phase difference between the satellite and the ground station
    """
    gs = wgs84.latlon(gt_loc[0], gt_loc[1])
    satellites = load.tle_file(tle_file)
    sat_by_name = {sat.name: sat for sat in satellites}
    sat = sat_by_name[satellite_name]

    ts = load.timescale()
    start_time_utc = datetime.fromtimestamp(sim_start_time+sat_start_sec, tz=pytz.utc)
    seconds_step_list = np.arange(start_time_utc.second, (start_time_utc.second+start_time_utc.microsecond*1e-6)+duration,step)
    t_steps = ts.utc(start_time_utc.year, start_time_utc.month, start_time_utc.day, start_time_utc.hour, start_time_utc.minute, seconds_step_list)
    el_az_gt_arr = []
    t_ = []

    for i,t in enumerate(t_steps):
        dist = (sat - gs).at(t)
        print(sat.at(t).position.km, sat.at(t).velocity.km_per_s, t.utc_datetime())
        from skyfield.positionlib import Geocentric as Geocentric   
        sat_pos = Geocentric([-2038.2120424471, 3733.1336847743, 5455.2396753985],[-7.1206988024, -2.4426043142, -0.9867188365])
        print("dist", dist)
        print("dist_", sat_pos-gs.at(t))
        
        alt, az, distance = dist.altaz()
        r = 1e8
        lam = 3e8/11.325e9
        el_gt = np.radians(alt.degrees)
        az_gt = np.radians(az.degrees)
        el_az_gt_arr.append([el_gt, az_gt])
        t_.append(sat_start_sec+i*step)
        
    # convert list to array
    el_az_gt_arr = np.array(el_az_gt_arr)
    t_ = np.array(t_)
    # print(el_az_gt_arr.shape)
    loc_3d = r*np.array([np.cos(el_az_gt_arr[:,0])*np.sin(el_az_gt_arr[:,1]), np.cos(el_az_gt_arr[:,0])*np.cos(el_az_gt_arr[:,1]), np.sin(el_az_gt_arr[:,0])]).T
    # print(loc_3d.shape)

    alpha = np.radians(rx_param['alpha'])
    beta = np.radians(rx_param['beta'])
    gamma = np.radians(rx_param['gamma'])
    d_10 = rx_param['d_10']
    d_12 = rx_param['d_12']

    loc_1 = np.array([0, 0, 0])
    loc_2 = d_10*np.array([np.cos(alpha)*np.cos(gamma), -np.sin(alpha)*np.sin(gamma), np.sin(gamma)])
    loc_0 = d_12*np.array([-np.sin(alpha)*np.cos(beta), -np.cos(alpha)*np.cos(beta), -np.sin(beta)])
    range_1 = np.linalg.norm(loc_3d-loc_1, axis=1)
    range_0 = np.linalg.norm(loc_3d-loc_0, axis=1)
    range_2 = np.linalg.norm(loc_3d-loc_2, axis=1)
    del_phi_01_gt = -(range_0-range_1)/lam*2*np.pi
    del_phi_12_gt = -(range_1-range_2)/lam*2*np.pi
    param_01_gt = np.polyfit(t_, del_phi_01_gt, 2)
    param_12_gt = np.polyfit(t_, del_phi_12_gt, 2)
    return t_, param_01_gt, param_12_gt
    # get the phase difference between the satellite and the ground station


In [3]:
def get_gt_locs(centeral_lat, central_lon, range_km, step_km):
    """
    Returns a list of ground truth locations in a square around the central_lat and central_lon
    :param centeral_lat: float - latitude of the central location
    :param central_lon: float - longitude of the central location
    :param range_km: tuple - range in km from the central location (-range_km, range_km)
    :param step_km: float - step size in km
    """
    r_earth = 6378.137 #km
    gt_locs = []
    lat_km_range = np.arange(-range_km, range_km+step_km, step_km)
    lon_km_range = np.arange(-range_km, range_km+step_km, step_km)
    
    for dx in lat_km_range:
        for dy in lon_km_range:
            new_lat  = centeral_lat  + (dy / r_earth) * (180 / np.pi)
            new_lon = central_lon + (dx / r_earth) * (180 / np.pi) / np.cos(new_lat * np.pi/180)
            gt_locs.append((new_lat, new_lon))
    return gt_locs

### Get observed phase difference

In [4]:
phase_comp = sio.loadmat('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/phase_comp.mat')
info = sio.loadmat('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/info.mat')

sat_dopp_track = phase_comp['dopp_track_dict'][0]
sat_id = phase_comp['id_dict'][0]

desired_sat = 0 # TODO: change this to the desired satellite
t_repeat = np.real(sat_dopp_track[desired_sat][:,0])
del_phi_repeat_01 = np.unwrap(np.angle(sat_dopp_track[desired_sat][:,2])-np.angle(sat_dopp_track[desired_sat][:,3]))
del_phi_repeat_12 = np.unwrap(np.angle(sat_dopp_track[desired_sat][:,3])-np.angle(sat_dopp_track[desired_sat][:,4]))

t_unique = np.unique(t_repeat)
# phase_diff calibration
ref_sig_idxes = info['ref_sig_idx']
ref_sig_idx_during_sat = np.where(((ref_sig_idxes[:,0]/info['sample_rate']>min(t_unique))&(ref_sig_idxes[:,0]/info['sample_rate']<max(t_unique)))[0])[0]
phase_offset_01 = info['phase_offset_01'][0,min(ref_sig_idx_during_sat)]
phase_offset_12 = -info['phase_offset_21'][0,min(ref_sig_idx_during_sat)]
print(phase_offset_01, phase_offset_12)
# fit 2nd order polynomial to phase difference
param_01_obs = np.polyfit(t_repeat, del_phi_repeat_01, 2)
param_12_obs = np.polyfit(t_repeat, del_phi_repeat_12, 2)

1.1068442897284978 0.812948240193633


### Get best rx_param from the sweep


In [5]:
alpha_list = np.arange(-4,4+0.2,0.2)
beta_list = np.arange(-4,4+0.2,0.2)
gamma_list = np.arange(-4,4+0.2,0.2)
d10_list = np.linspace(13.25e-2-0.2e-2,13.25e-2+0.2e-2,10)
d12_list = np.linspace(13.25e-2-0.2e-2,13.25e-2+0.2e-2,10)


param = sio.loadmat('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/sweep_param_res.mat')
best_param_idx = np.unravel_index(np.argmin(param['slope_err']), param['slope_err'].shape)
print(best_param_idx, param['slope_err'][best_param_idx], param['slope_err'][best_param_idx[0], best_param_idx[1], best_param_idx[2], best_param_idx[3], best_param_idx[4], best_param_idx[5], best_param_idx[6]])
print(alpha_list[best_param_idx[0]], beta_list[best_param_idx[1]], gamma_list[best_param_idx[2]], d10_list[best_param_idx[3]], d12_list[best_param_idx[4]])

(6, 7, 0, 0, 6, 2, 0) 2.6827814227678003e-09 2.6827814227678003e-09
-2.799999999999999 -2.5999999999999988 -4.0 0.1305 0.13316666666666668


### Get phase difference at different locations and compare with the observed phase difference

In [17]:
desired_sat_id = 'STARLINK-2377'#sat_id[desired_sat][0]
# start_time = info['start_time'][0][0]
start_time = 1708466442.0
sat_start_sec = 0 #np.floor(np.min(np.real(t_repeat)))
duration = 600#np.ceil(np.max(np.real(t_repeat))-np.min(np.real(t_repeat)))
print(start_time, sat_start_sec, duration)

rx_param = {'alpha': 0, 'beta': 0.3, 'gamma': 1.22, 'd_10': 13.05e-2, 'd_12': 13.09e-2}
# get ground stations locations in a square around the central location
gt_locs = get_gt_locs(+40.072969193609126, -88.20958051030674, 25, 1)

diff_01_arr = []
diff_12_arr = []
f = open('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/sim.out', 'w')

for i,loc in enumerate(gt_locs):
    print(i,loc, file=f, flush=True)
    t, param_01_gt, param_12_gt = get_simulated_phase_difference(gt_loc = loc, rx_param=rx_param, tle_file='/home/ishani/LeoPNT/USRP_data/2_20/utctime_20Feb2024_22_00_53_starlink.tle', 
                                    satellite_name=desired_sat_id, sim_start_time=start_time, sat_start_sec=sat_start_sec,duration=duration, step=60)
    phase_diff_01_gt = np.polyval(param_01_gt, t_unique) 
    phase_diff_12_gt = np.polyval(param_12_gt, t_unique)
    phase_diff_01_obs = np.polyval(param_01_obs, t_unique) 
    phase_diff_12_obs = np.polyval(param_12_obs, t_unique)

    diff_01 = np.mean(phase_diff_01_gt - phase_diff_01_obs)
    diff_12 = np.mean(phase_diff_12_gt - phase_diff_12_obs)
    diff_01_arr.append(diff_01)
    diff_12_arr.append(diff_12)

diff_01_arr = np.array(diff_01_arr)
diff_12_arr = np.array(diff_12_arr)
# save to file
np.save('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/diff_01_arr.npy', diff_01_arr)
np.save('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/diff_12_arr.npy', diff_12_arr)
np.save('/home/ishani/LeoPNT/starlink_ofdm/Starloc/5Feb2024_23-58-08/gt_locs.npy', gt_locs)
# get minimum index
min_idx_01 = np.argmin(diff_01_arr)
min_idx_12 = np.argmin(diff_12_arr)
# get loc at min index
min_loc_01 = gt_locs[min_idx_01]
min_loc_12 = gt_locs[min_idx_12]
# print to file
print(min_loc_01, min_loc_12, file=f)
f.close()

1708466442.0 0 600
[-2038.88091592  3732.56701826  5455.32529266] [-7.12041183 -2.44313955 -0.98717866] 2024-02-20 22:00:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.8484 N longitude -88.5021 E elevation 0.0 m target=-147797>
dist_ <Geometric ICRS position and velocity center=WGS84 latitude +39.8484 N longitude -88.5021 E elevation 0.0 m>
[-2461.39215916  3578.01972287  5384.31819171] [-6.95823861 -2.70659018 -1.3788833 ] 2024-02-20 22:01:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.8484 N longitude -88.5021 E elevation 0.0 m target=-147797>
dist_ <Geometric ICRS position and velocity center=WGS84 latitude +39.8484 N longitude -88.5021 E elevation 0.0 m>
[-2873.26685886  3408.01025758  5289.97904868] [-6.76598498 -2.95836009 -1.76463611] 2024-02-20 22:02:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.8484 N longitude -88.5021 E elevation 0.0 m target=-147797>
d

[-2038.88091592  3732.56701826  5455.32529266] [-7.12041183 -2.44313955 -0.98717866] 2024-02-20 22:00:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.9203 N longitude -88.5024 E elevation 0.0 m target=-147797>
dist_ <Geometric ICRS position and velocity center=WGS84 latitude +39.9203 N longitude -88.5024 E elevation 0.0 m>
[-2461.39215916  3578.01972287  5384.31819171] [-6.95823861 -2.70659018 -1.3788833 ] 2024-02-20 22:01:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.9203 N longitude -88.5024 E elevation 0.0 m target=-147797>
dist_ <Geometric ICRS position and velocity center=WGS84 latitude +39.9203 N longitude -88.5024 E elevation 0.0 m>
[-2873.26685886  3408.01025758  5289.97904868] [-6.76598498 -2.95836009 -1.76463611] 2024-02-20 22:02:42+00:00
dist <Geometric ICRS position and velocity at date t center=WGS84 latitude +39.9203 N longitude -88.5024 E elevation 0.0 m target=-147797>
dist_ <Geometric ICR

KeyboardInterrupt: 

In [27]:
# convert yyyyDOYhhmmss.sss to datetime timestamp
def get_timestamp(yyyyDOYhhmmss):
    yyyy = int(yyyyDOYhhmmss[0:4])
    DOY = int(yyyyDOYhhmmss[4:7])
    hh = int(yyyyDOYhhmmss[7:9])
    mm = int(yyyyDOYhhmmss[9:11])
    ss = int(yyyyDOYhhmmss[11:13])
    print(yyyy, DOY, hh, mm, ss)
    return datetime(yyyy, 1, 1, hh, mm, ss, tzinfo=pytz.utc) + timedelta(DOY - 1)

# get the timestamp
start_time = get_timestamp('2024051220042.000')
print(start_time)
# convert to timestamp
from datetime import tzinfo
import time
start_timestamp = datetime.timestamp(start_time)
print(start_timestamp)

2024 51 22 0 42
2024-02-20 22:00:42+00:00
1708466442.0


In [40]:
# utc epoch timestamp to datetime
print(datetime.fromtimestamp(start_timestamp, tz=pytz.utc))

2024-02-20 16:20:42+00:00


In [12]:
from skyfield.positionlib import ICRF, Geocentric
from skyfield.api import load, wgs84
ts = load.timescale()
a = Geocentric([1,1,1])
a.velocity = [2,2,2]
a.t = 1
print(a)


<Geocentric ICRS position and velocity at date t center=399>


In [15]:
from skyfield.api import load, wgs84
from skyfield.units import Distance, Velocity,
from skyfield.positionlib import ICRF, Geocentric
ts = load.timescale()
t = ts.utc(2024, 2, 20, 22, 0, 42)
pos = Distance(km = [-2038.2120424471, 3733.1336847743, 5455.2396753985])
sat_pos = Geocentric(,[-7.1206988024, -2.4426043142, -0.9867188365])
print(sat_pos)
gs = wgs84.latlon(+40.072969193609126, -88.20958051030674)
sat_pos.t = t
dist = (sat_pos - gs).at(t)
print("dist", dist)
print("dist_", sat_pos-gs.at(t))

alt, az, distance = dist.altaz()

<Geocentric ICRS position and velocity center=399>


AttributeError: 'GeographicPosition' object has no attribute 'position'